In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

scenario_score_path = (
    project_root
    / "data"
    / "processed"
    / "la_preliminary_opportunity_scores.csv"
)

competition_sensitivity_path = (
    project_root
    / "data"
    / "processed"
    / "la_competition_definition_sensitivity_scores.csv"
)

scenario_scores = pd.read_csv(
    scenario_score_path,
    dtype={"GEOID": "string"}
)

competition_scores = pd.read_csv(
    competition_sensitivity_path,
    dtype={"GEOID": "string"}
)

scenario_columns = [
    "GEOID",
    "demand_focused_rank",
    "balanced_rank",
    "competition_focused_rank",
    "top_20_scenario_count"
]

final_candidates = competition_scores.merge(
    scenario_scores[scenario_columns],
    on="GEOID",
    how="left",
    validate="one_to_one"
)

print("Competition sensitivity rows:", len(competition_scores))
print("Final merged rows:", len(final_candidates))
print(
    "Duplicate GEOIDs:",
    final_candidates["GEOID"].duplicated().sum()
)
print(
    "Missing scenario results:",
    final_candidates["top_20_scenario_count"].isna().sum()
)

display(
    final_candidates[
        [
            "GEOID",
            "baseline_rank",
            "expanded_rank",
            "competition_robustness",
            "demand_focused_rank",
            "balanced_rank",
            "competition_focused_rank",
            "top_20_scenario_count"
        ]
    ].head(10)
)

Competition sensitivity rows: 964
Final merged rows: 964
Duplicate GEOIDs: 0
Missing scenario results: 0


,GEOID,baseline_rank,expanded_rank,competition_robustness,demand_focused_rank,balanced_rank,competition_focused_rank,top_20_scenario_count
0,06037206010,372,364,outside_top_20,289.0,372.0,499.0,0
1,06037206020,885,880,outside_top_20,927.0,885.0,923.0,0
2,06037108203,75,64,outside_top_20,98.0,75.0,50.0,0
3,06037920121,136,123,outside_top_20,236.0,136.0,111.0,0
4,06037135204,532,525,outside_top_20,593.0,532.0,389.0,0
5,06037460101,401,393,outside_top_20,500.0,401.0,231.0,0
6,06037207400,647,649,outside_top_20,560.0,647.0,768.0,0
7,06037192410,601,617,outside_top_20,620.0,601.0,711.0,0
8,06037195100,330,361,outside_top_20,427.0,330.0,290.0,0
9,06037195300,829,833,outside_top_20,823.0,829.0,882.0,0


In [3]:
# Identify tracts that passed each stability test
final_candidates["stable_across_weights"] = (
    final_candidates["top_20_scenario_count"] >= 2
)

final_candidates["stable_across_competition_definitions"] = (
    final_candidates["competition_robustness"]
    == "stable_top_20"
)

final_candidates["final_candidate_status"] = np.select(
    [
        (
            final_candidates["stable_across_weights"]
            & final_candidates[
                "stable_across_competition_definitions"
            ]
        ),
        (
            final_candidates["stable_across_weights"]
            & ~final_candidates[
                "stable_across_competition_definitions"
            ]
        ),
        (
            ~final_candidates["stable_across_weights"]
            & final_candidates[
                "stable_across_competition_definitions"
            ]
        )
    ],
    [
        "robust_finalist",
        "weight_stable_only",
        "competition_stable_only"
    ],
    default="not_shortlisted"
)

# Average rank across four alternative specifications
rank_columns = [
    "demand_focused_rank",
    "balanced_rank",
    "competition_focused_rank",
    "expanded_rank"
]

final_candidates["average_scenario_rank"] = (
    final_candidates[rank_columns]
    .mean(axis=1)
    .round(2)
)

final_candidates["consensus_rank"] = (
    final_candidates["average_scenario_rank"]
    .rank(ascending=True, method="min")
    .astype("int64")
)

status_summary = (
    final_candidates["final_candidate_status"]
    .value_counts()
    .rename_axis("final_candidate_status")
    .reset_index(name="tract_count")
)

display(status_summary)

robust_finalists = (
    final_candidates.loc[
        final_candidates["final_candidate_status"]
        == "robust_finalist"
    ]
    .sort_values("average_scenario_rank")
    .copy()
)

print("Robust finalists:", len(robust_finalists))

display(
    robust_finalists[
        [
            "GEOID",
            "chinese_total_estimate",
            "chinese_share_pct",
            "median_household_income",
            "competitors_within_3_miles",
            "expanded_competitors_within_3_miles",
            "demand_focused_rank",
            "balanced_rank",
            "competition_focused_rank",
            "expanded_rank",
            "average_scenario_rank",
            "consensus_rank"
        ]
    ]
)

,final_candidate_status,tract_count
0,not_shortlisted,944
1,robust_finalist,17
2,weight_stable_only,3


Robust finalists: 17


,GEOID,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_3_miles,expanded_competitors_within_3_miles,demand_focused_rank,balanced_rank,competition_focused_rank,expanded_rank,average_scenario_rank,consensus_rank
680,06037403407,1661,70.14,156552.0,6,10,2.0,1.0,2.0,1,1.50,1
95,06037403325,2735,58.12,111445.0,2,7,1.0,4.0,1.0,5,2.75,2
675,06037430400,1674,39.66,182292.0,8,20,4.0,2.0,4.0,6,4.00,3
662,06037403404,1035,48.96,173289.0,7,10,8.0,5.0,6.0,2,5.25,4
66,06037408503,1943,29.42,145921.0,7,8,7.0,8.0,9.0,3,6.75,5
766,06037670413,881,17.99,214625.0,0,0,14.0,7.0,3.0,4,7.00,6
422,06037403324,2944,45.87,114547.0,6,13,3.0,10.0,7.0,9,7.25,7
295,06037464102,2617,57.62,239052.0,16,39,6.0,3.0,23.0,7,9.75,8
815,06037670326,680,21.36,250001.0,3,7,17.0,13.0,5.0,8,10.75,9
74,06037408703,2644,49.64,127500.0,10,26,5.0,11.0,16.0,15,11.75,10


In [4]:
robust_finalists["finalist_rank"] = (
    robust_finalists["average_scenario_rank"]
    .rank(ascending=True, method="min")
    .astype("int64")
)

robust_finalists["priority_tier"] = np.select(
    [
        robust_finalists["finalist_rank"] <= 5,
        robust_finalists["finalist_rank"] <= 10
    ],
    [
        "priority_candidate",
        "secondary_candidate"
    ],
    default="reserve_candidate"
)

tier_summary = (
    robust_finalists["priority_tier"]
    .value_counts()
    .rename_axis("priority_tier")
    .reset_index(name="tract_count")
)

display(tier_summary)

display(
    robust_finalists[
        [
            "finalist_rank",
            "GEOID",
            "chinese_total_estimate",
            "chinese_share_pct",
            "median_household_income",
            "competitors_within_3_miles",
            "expanded_competitors_within_3_miles",
            "average_scenario_rank",
            "priority_tier"
        ]
    ].sort_values("finalist_rank")
)

,priority_tier,tract_count
0,reserve_candidate,7
1,priority_candidate,5
2,secondary_candidate,5


,finalist_rank,GEOID,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_3_miles,expanded_competitors_within_3_miles,average_scenario_rank,priority_tier
680,1,06037403407,1661,70.14,156552.0,6,10,1.50,priority_candidate
95,2,06037403325,2735,58.12,111445.0,2,7,2.75,priority_candidate
675,3,06037430400,1674,39.66,182292.0,8,20,4.00,priority_candidate
662,4,06037403404,1035,48.96,173289.0,7,10,5.25,priority_candidate
66,5,06037408503,1943,29.42,145921.0,7,8,6.75,priority_candidate
766,6,06037670413,881,17.99,214625.0,0,0,7.00,secondary_candidate
422,7,06037403324,2944,45.87,114547.0,6,13,7.25,secondary_candidate
295,8,06037464102,2617,57.62,239052.0,16,39,9.75,secondary_candidate
815,9,06037670326,680,21.36,250001.0,3,7,10.75,secondary_candidate
74,10,06037408703,2644,49.64,127500.0,10,26,11.75,secondary_candidate


In [5]:
robust_finalists["priority_tier"] = (
    robust_finalists["priority_tier"].replace(
        {
            "priority_candidate": "priority_candidate_area",
            "secondary_candidate": "secondary_candidate_area",
            "reserve_candidate": "reserve_candidate_area"
        }
    )
)

# Load tract geometry
tract_geometry_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_site_selection_inputs.gpkg"
)

tract_geometry = gpd.read_file(
    tract_geometry_path
)[["GEOID", "geometry"]]

# Attach geometry to the 17 candidate areas
final_candidate_areas = gpd.GeoDataFrame(
    robust_finalists.merge(
        tract_geometry,
        on="GEOID",
        how="left",
        validate="one_to_one"
    ),
    geometry="geometry",
    crs=tract_geometry.crs
)

final_areas_csv_path = (
    project_root
    / "data"
    / "processed"
    / "la_final_candidate_areas.csv"
)

final_areas_gpkg_path = (
    project_root
    / "data"
    / "processed"
    / "la_final_candidate_areas.gpkg"
)

final_candidate_areas.drop(
    columns="geometry"
).sort_values(
    "finalist_rank"
).to_csv(
    final_areas_csv_path,
    index=False
)

final_candidate_areas.to_file(
    final_areas_gpkg_path,
    layer="final_candidate_areas",
    driver="GPKG"
)

print("Final candidate areas:", len(final_candidate_areas))
print(
    "Missing geometry:",
    final_candidate_areas.geometry.isna().sum()
)
print(
    "Duplicate GEOIDs:",
    final_candidate_areas["GEOID"].duplicated().sum()
)
print("CSV exists:", final_areas_csv_path.exists())
print("GeoPackage exists:", final_areas_gpkg_path.exists())
print("GeoPackage path:", final_areas_gpkg_path)

Final candidate areas: 17
Missing geometry: 0
Duplicate GEOIDs: 0
CSV exists: True
GeoPackage exists: True
GeoPackage path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_final_candidate_areas.gpkg


In [6]:
top_5_areas = (
    robust_finalists.loc[
        robust_finalists["finalist_rank"] <= 5
    ]
    .sort_values("finalist_rank")
    .copy()
)

top_5_summary = top_5_areas[
    [
        "finalist_rank",
        "GEOID",
        "chinese_total_estimate",
        "chinese_share_pct",
        "median_household_income",
        "competitors_within_1_mile",
        "expanded_competitors_within_1_mile",
        "competitors_within_3_miles",
        "expanded_competitors_within_3_miles",
        "average_scenario_rank"
    ]
].copy()

top_5_summary["recommendation"] = (
    "Priority area for site-level evaluation"
)

top_5_path = (
    project_root
    / "data"
    / "processed"
    / "la_priority_candidate_areas_top5.csv"
)

top_5_summary.to_csv(
    top_5_path,
    index=False
)

print("Priority candidate areas:", len(top_5_summary))
print("Duplicate GEOIDs:", top_5_summary["GEOID"].duplicated().sum())
print("Top-5 CSV exists:", top_5_path.exists())
print("Top-5 CSV path:", top_5_path)

display(
    top_5_summary.style.format(
        {
            "chinese_share_pct": "{:.2f}%",
            "median_household_income": "${:,.0f}",
            "average_scenario_rank": "{:.2f}"
        }
    )
)

Priority candidate areas: 5
Duplicate GEOIDs: 0
Top-5 CSV exists: True
Top-5 CSV path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_priority_candidate_areas_top5.csv


,finalist_rank,GEOID,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_1_mile,expanded_competitors_within_1_mile,competitors_within_3_miles,expanded_competitors_within_3_miles,average_scenario_rank,recommendation
680,1,06037403407,1661,70.14%,"$156,552",0,0,6,10,1.50,Priority area for site-level evaluation
95,2,06037403325,2735,58.12%,"$111,445",0,0,2,7,2.75,Priority area for site-level evaluation
675,3,06037430400,1674,39.66%,"$182,292",0,0,8,20,4.00,Priority area for site-level evaluation
662,4,06037403404,1035,48.96%,"$173,289",0,0,7,10,5.25,Priority area for site-level evaluation
66,5,06037408503,1943,29.42%,"$145,921",0,0,7,8,6.75,Priority area for site-level evaluation


## Final Candidate-Area Screening Results

The analysis identified 964 Census tracts that met the demographic-data
quality requirements for preliminary screening.

Two sensitivity tests were used:

1. Alternative weighting scenarios for demand, income, and competition.
2. Alternative competitor definitions using 510 confirmed high-confidence
   candidates and 772 expanded competitor candidates.

Seventeen Census tracts remained strong across both sensitivity tests.
These areas were divided into:

- 5 priority candidate areas
- 5 secondary candidate areas
- 7 reserve candidate areas

Census tract `06037403407` received the strongest overall consensus ranking.

These results identify areas where site-level investigation should be
prioritized. They do not represent specific available restaurant properties.
Final site selection would require additional information such as commercial
rent, parcel availability, traffic, parking, accessibility, zoning, and
on-the-ground market validation.